# L2_T1 — Data Gathering: Skin Condition Analysis Datasets
### Layer 2 of the Aura pipeline · Task 1 of N

**What this notebook does:** gathers the datasets for Layer 2 (acne, hyperpigmentation, dryness/oiliness,
pore visibility, sensitivity) using the same safe storage pattern the Layer 1 series was rebuilt around:
everything is written to Colab's local disk first, bundled into single `.tar.gz` archives, copied to Drive as
one file each, and verified from a fresh mount before anything is trusted.

**Correction to earlier research — read this first.** The original Layer 2 dataset research named
`khanusa/facial-skin-conditions` (Hugging Face) as the primary training set. **That dataset does not exist** —
that Hugging Face account only has a text-to-speech model. A second candidate from the same research (a Kaggle
dataset under the name `killa92`) also could not be found. Neither was verified by direct fetch at the time,
unlike the Layer 1 datasets. Both are dropped. This notebook uses only sources that were re-verified.

**What was actually verified, and how each one is obtained:**

| Dataset | What it gives Layer 2 | How it's obtained here |
|---|---|---|
| DermNet (HF mirror `rkdeva/Dermnet-Train`, 15,297 images) | Broad skin-condition images, disease-level labels | **Scripted** — Hugging Face `datasets` |
| ACNE04 (ICCV 2019, Wu et al.) | Acne severity grading + lesion counts | **Manual** — authors host it on Google Drive / Baidu |
| Oily / Dry / Normal Skin Types (Kaggle, ~126 MB) | Skin-type (oiliness/dryness) labels | **Manual** — Kaggle download |
| SkinCon (NeurIPS 2022, Stanford) | 48 dermatologist concepts incl. redness/erythema (sensitivity) | **Manual / gated** — request via SkinCon site |
| DDI (Stanford AIMI, 656 images) | Biopsy-confirmed, diverse skin tones | **Manual / gated** — Stanford AIMI registration |

**Honest scope note:** none of these map cleanly onto Aura's cosmetic taxonomy. DermNet is disease-oriented,
ACNE04 is acne only, the skin-type set is whole-image rather than per-zone, and nothing here labels **pore
visibility** at all. Layer 2's per-zone, multi-condition design will need the same kind of in-house,
dermatologist-labeled data Layer 1 does — these public sets are a starting point, not a full solution.

## Step 0 — Mount Drive and set up local + Drive folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition"
DRIVE_DATASETS = f"{DRIVE_ROOT}/datasets"
DRIVE_DERMNET = f"{DRIVE_DATASETS}/dermnet"
MANUAL_DATASETS = ["acne04", "skin_types", "skincon", "ddi"]
DRIVE_MANUAL_INCOMING = {name: f"{DRIVE_DATASETS}/{name}/_incoming" for name in MANUAL_DATASETS}
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.json"

LOCAL_ROOT = "/content/aura_local/Layer2_SkinCondition"
LOCAL_DERMNET = f"{LOCAL_ROOT}/datasets/dermnet"

for d in [DRIVE_ROOT, DRIVE_DATASETS, DRIVE_DERMNET, LOCAL_ROOT, LOCAL_DERMNET, *DRIVE_MANUAL_INCOMING.values()]:
    os.makedirs(d, exist_ok=True)

import shutil
_, _, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free: {free/1e9:.1f} GB")
_, _, lfree = shutil.disk_usage("/content")
print(f"Colab local disk free: {lfree/1e9:.1f} GB")


Mounted at /content/drive
Drive free: 46.7 GB
Colab local disk free: 75.3 GB


## Step 1 — Record licensing/provenance before downloading anything

Aura is a commercial venture, so "free to download" is not the same as "free to use." This is recorded now,
next to the data, so it's not rediscovered later. **These are what the sources themselves state or what could
not be confirmed — not legal advice, and worth a real check before any commercial use.**

In [2]:
notice = """# Layer 2 data provenance notice (L2_T1)

## DermNet (Hugging Face mirror: rkdeva/Dermnet-Train)
- Original source: DermNet Skin Disease Atlas (~23,000 images, 23 super-classes).
- The Hugging Face mirror's dataset card says only "More Information needed" -- NO license is stated there.
- Treat as UNVERIFIED for commercial use. Check DermNet's own image-use terms before relying on it beyond research.

## ACNE04 (Wu et al., ICCV 2019, github.com/xpwu95/LDL)
- Authors state: "free for academic usage. For other purposes, please contact Xiaoping Wu."
- That means commercial use needs the authors' permission. Do not assume it is cleared for Aura's product.

## Oily / Dry / Normal Skin Types (Kaggle)
- License not confirmed in this session. Read the license on the Kaggle dataset page when you download it.

## SkinCon (Daneshjou et al., NeurIPS 2022) and DDI (Stanford AIMI)
- Both are gated / request-based. Their own access terms apply -- read them when you apply.
- SkinCon images come from Fitzpatrick17k (already used in Layer 1) and DDI.

## General
- None of these were verified as cleared for commercial redistribution or for training a commercial model.
- Before V4 (public beta), get a real license review of every dataset the shipped model was trained on.
"""
path = f"{DRIVE_ROOT}/DATA_PROVENANCE_NOTICE.md"
with open(path, "w") as f:
    f.write(notice)
print(f"Written: {path}")


Written: /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/DATA_PROVENANCE_NOTICE.md


## Step 2 — Install dependencies and define the archive/upload/verify helper

In [3]:
!pip install -q datasets pillow pandas tqdm

import tarfile, time

def archive_and_upload(local_dir, archive_filename, drive_dest_dir, arcname=None):
    """Tar locally, copy the ONE archive to Drive, verify size. Same pattern as the Layer 1 series."""
    local_archive_path = os.path.join("/content/aura_local", archive_filename)
    arcname = arcname or os.path.basename(local_dir.rstrip("/"))
    print(f"Archiving {local_dir} ...")
    with tarfile.open(local_archive_path, "w:gz") as tar:
        tar.add(local_dir, arcname=arcname)
    local_size = os.path.getsize(local_archive_path)
    print(f"  Local archive: {local_size/1e6:.1f} MB")

    drive_archive_path = os.path.join(drive_dest_dir, archive_filename)
    print(f"Copying to Drive: {drive_archive_path} ...")
    shutil.copy2(local_archive_path, drive_archive_path)
    time.sleep(2)
    drive_size = os.path.getsize(drive_archive_path)
    if drive_size != local_size:
        raise RuntimeError(f"SIZE MISMATCH for {archive_filename}: local={local_size}, drive={drive_size}")
    print(f"  ✅ Verified on Drive: {drive_size/1e6:.1f} MB matches local exactly.")
    return drive_archive_path, drive_size


## Step 3 — DermNet: load and INSPECT before exporting anything

Same inspect-before-assume discipline as `L1_T4`: print the real schema and the real label names first. The
label vocabulary isn't something to guess at — which labels map to Aura's conditions (acne, pigmentation,
redness) gets decided from what's actually printed here, in a later notebook.

In [4]:
from datasets import load_dataset

dermnet = load_dataset("rkdeva/Dermnet-Train")
print(dermnet)
print("\nFeatures:", dermnet["train"].features)


README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

data/train-00000-of-00003-edd79f1dbb5482(…): reconstructing file:   0%|          |  0.00B /  484MB            

data/train-00000-of-00003-edd79f1dbb5482(…): downloading bytes:           |  0.00B            

data/train-00001-of-00003-c65afe3db30225(…): reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00001-of-00003-c65afe3db30225(…): downloading bytes:           |  0.00B            

data/train-00002-of-00003-a839422ecfea75(…): reconstructing file:   0%|          |  0.00B /  482MB            

data/train-00002-of-00003-a839422ecfea75(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15297 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 15297
    })
})

Features: {'image': Image(mode=None, decode=True), 'label': Value('string')}


In [5]:
import pandas as pd

ds = dermnet["train"]
label_feature = ds.features["label"]

def label_to_str(lab):
    if isinstance(lab, int) and hasattr(label_feature, "names"):
        return label_feature.names[lab]
    return str(lab)

all_labels = [label_to_str(l) for l in ds["label"]]
label_counts = pd.Series(all_labels).value_counts()
print(f"{len(label_counts)} distinct labels across {len(ds)} images:\n")
print(label_counts.to_string())


865 distinct labels across 15297 images:

molluscum contagiosum                          196
lichen planus                                  195
alopecia areata                                161
herpes type  recurrent                         131
acne cystic                                    120
IMG                                            119
vasculitis                                     113
eczema fingertips                              112
lupus chronic cutaneous                        110
perioral dermatitis                            109
genital warts                                  107
rosacea                                        106
basal cell carcinoma lesion                    105
seborrheic keratoses ruff                      105
psoriasis palms soles                          105
herpes type  primary                           105
distal subungual onychomycosis                 105
scabies                                        105
eczema nummular                         

**Look at the label list above before continuing.** Which of these correspond to Aura's five conditions
(acne, hyperpigmentation, dryness/oiliness, pore visibility, sensitivity) is a judgment call to make from the
real names printed here. Expect it to be partial: DermNet labels *diseases*, not cosmetic skin-state traits,
so some of Aura's categories (pore visibility especially) won't have a direct DermNet equivalent at all.

## Step 4 — Export DermNet to local disk, then archive + upload + verify

In [6]:
import csv
from tqdm.auto import tqdm

img_dir = os.path.join(LOCAL_DERMNET, "train")
os.makedirs(img_dir, exist_ok=True)
csv_path = os.path.join(LOCAL_DERMNET, "train_labels.csv")

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "label"])
    for i in tqdm(range(len(ds)), desc="Exporting DermNet (local disk)"):
        ex = ds[i]
        fname = f"{i:06d}.jpg"
        fpath = os.path.join(img_dir, fname)
        if not os.path.exists(fpath):
            ex["image"].convert("RGB").save(fpath, "JPEG", quality=92)
        writer.writerow([fname, label_to_str(ex["label"])])

n_exported = len([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
print(f"Exported {n_exported} / {len(ds)} images")
assert n_exported == len(ds), "Export count mismatch -- do not archive yet."


Exporting DermNet (local disk):   0%|          | 0/15297 [00:00<?, ?it/s]

Exported 15297 / 15297 images


In [7]:
dermnet_drive_path, dermnet_drive_size = archive_and_upload(
    local_dir=LOCAL_DERMNET,
    archive_filename="dermnet.tar.gz",
    drive_dest_dir=DRIVE_DERMNET,
    arcname="dermnet",
)


Archiving /content/aura_local/Layer2_SkinCondition/datasets/dermnet ...
  Local archive: 1559.6 MB
Copying to Drive: /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/dermnet/dermnet.tar.gz ...
  ✅ Verified on Drive: 1559.6 MB matches local exactly.


## Step 5 — The manual / gated datasets: check what's on Drive

None of these four can be scripted honestly, so this is the same pattern as MST-E in the Layer 1 series: you
fetch each one yourself, upload the **archive as-is** (single-file uploads don't have the many-small-files
sync problem) into its `_incoming` folder on Drive, and this cell reports what's there. **Nothing here is
required to proceed** — absent ones are simply recorded as not-yet-downloaded.

| Dataset | Where to get it | Notes |
|---|---|---|
| `acne04` | github.com/xpwu95/LDL → Google Drive or Baidu link | Academic-use license (see Step 1). Also see github.com/AIpourlapeau/acne04v2 for a cleaned 1,204-image relabel with lesion annotations |
| `skin_types` | Kaggle — search "Oily, Dry and Normal Skin Types Dataset" (~126 MB zip) | Confirm the exact page and its license yourself |
| `skincon` | skincon-dataset.github.io | Request-based; annotations map onto Fitzpatrick17k images you already have from Layer 1 |
| `ddi` | ddi-dataset.github.io / Stanford AIMI | Registration-gated; 656 images |

In [8]:
manual_status = {}
for name, incoming in DRIVE_MANUAL_INCOMING.items():
    files = [f for f in os.listdir(incoming) if not f.startswith(".")]
    if files:
        info = [(f, os.path.getsize(os.path.join(incoming, f)) / 1e6) for f in files]
        manual_status[name] = {"exists": True, "files": [{"name": n, "size_mb": round(s, 1)} for n, s in info]}
        print(f"✅ {name}: " + ", ".join(f"{n} ({s:.1f} MB)" for n, s in info))
    else:
        manual_status[name] = {"exists": False}
        print(f"⬜ {name}: not downloaded yet -- {incoming}")
print("\nThese are left as single archives on Drive and never extracted here (extracting onto Drive would")
print("recreate the many-small-files problem this series was rebuilt to avoid).")


⬜ acne04: not downloaded yet -- /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/acne04/_incoming
⬜ skin_types: not downloaded yet -- /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/skin_types/_incoming
⬜ skincon: not downloaded yet -- /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/skincon/_incoming
⬜ ddi: not downloaded yet -- /content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/ddi/_incoming

These are left as single archives on Drive and never extracted here (extracting onto Drive would
recreate the many-small-files problem this series was rebuilt to avoid).


## Step 6 — Force a full Drive sync and re-verify from a fresh mount

In [9]:
drive.flush_and_unmount()
print("Drive unmounted -- forced sync of pending writes.")
time.sleep(3)
drive.mount('/content/drive')

path = os.path.join(DRIVE_DERMNET, "dermnet.tar.gz")
if os.path.exists(path) and os.path.getsize(path) == dermnet_drive_size:
    print(f"✅ Confirmed after fresh remount: dermnet.tar.gz ({os.path.getsize(path)/1e6:.1f} MB)")
    all_ok = True
else:
    print("❌ dermnet.tar.gz missing or wrong size after remount -- do NOT close this session; re-run Step 4.")
    all_ok = False


Drive unmounted -- forced sync of pending writes.
Mounted at /content/drive
✅ Confirmed after fresh remount: dermnet.tar.gz (1559.6 MB)


In [10]:
import tarfile as _tf

# Integrity check: count members and fully read the last one (catches truncation)
with _tf.open(path, "r:gz") as tar:
    members = tar.getmembers()
    jpgs = [m for m in members if m.name.endswith(".jpg")]
    csvs = [m for m in members if m.name.endswith(".csv")]
    last = tar.extractfile(members[-1])
    _ = last.read() if last else b""
print(f"Archive members: {len(members)} | .jpg: {len(jpgs)} | .csv: {len(csvs)}")
assert len(jpgs) == len(ds), f"Expected {len(ds)} images in the archive, found {len(jpgs)}"
print("✅ Archive structurally complete.")


Archive members: 15300 | .jpg: 15297 | .csv: 1
✅ Archive structurally complete.


## Step 7 — Write the manifest (last, only after verification passed)

In [11]:
import json
from datetime import datetime, timezone

manifest = {
    "layer": "02_skin_condition_analysis",
    "task": "L2_T1_data_gathering",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "storage_pattern": "One .tar.gz per scripted dataset on Drive; manual datasets kept as single uploaded "
                       "archives in _incoming/ and never extracted on Drive.",
    "correction_note": "khanusa/facial-skin-conditions (named in the original Layer 2 research) does not exist; "
                       "a second candidate (killa92 on Kaggle) could not be verified. Both were dropped.",
    "datasets": {
        "dermnet": {
            "source": "huggingface rkdeva/Dermnet-Train (train split only; no test split verified)",
            "drive_archive": path,
            "archive_size_mb": round(dermnet_drive_size / 1e6, 1),
            "image_count": len(ds),
            "label_count": int(len(label_counts)),
            "license": "UNVERIFIED -- mirror's dataset card states none",
        },
        **{name: {**status, "incoming_dir": DRIVE_MANUAL_INCOMING[name]} for name, status in manual_status.items()},
    },
    "provenance_notice": f"{DRIVE_ROOT}/DATA_PROVENANCE_NOTICE.md",
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


{
  "layer": "02_skin_condition_analysis",
  "task": "L2_T1_data_gathering",
  "generated_at_utc": "2026-10-08T03:28:45.659472+00:00",
  "storage_pattern": "One .tar.gz per scripted dataset on Drive; manual datasets kept as single uploaded archives in _incoming/ and never extracted on Drive.",
  "correction_note": "khanusa/facial-skin-conditions (named in the original Layer 2 research) does not exist; a second candidate (killa92 on Kaggle) could not be verified. Both were dropped.",
  "datasets": {
    "dermnet": {
      "source": "huggingface rkdeva/Dermnet-Train (train split only; no test split verified)",
      "drive_archive": "/content/drive/MyDrive/Projects/Aura/Layer2_SkinCondition/datasets/dermnet/dermnet.tar.gz",
      "archive_size_mb": 1559.6,
      "image_count": 15297,
      "label_count": 865,
      "license": "UNVERIFIED -- mirror's dataset card states none"
    },
    "acne04": {
      "exists": false,
      "incoming_dir": "/content/drive/MyDrive/Projects/Aura/Layer2_S

## Done — what to do next

- DermNet is safely archived and verified on Drive. The four manual datasets are optional for now — download
  whichever you can; `L2_T2` can start with DermNet alone and fold the others in as they arrive.
- **Before building on any of this commercially:** the provenance notice written in Step 1 lists what could not
  be confirmed. ACNE04 in particular is academic-use-only unless the authors grant permission.
- **The structural gap to plan for now, not later:** nothing gathered here labels pore visibility, and nothing
  is zone-annotated (forehead / cheeks / nose / chin) — which is how Layer 2 is specified. Closing that needs
  the same in-house, dermatologist-labeled data Layer 1 is waiting on. Worth designing the V2 contribution
  flow so one consented photo can carry both Layer 1 labels (tone, undertone) and Layer 2 labels (per-zone
  conditions) instead of collecting twice.
- Next notebook: **`L2_T2` — Label mapping + preprocessing:** decide which DermNet labels correspond to
  Aura's conditions (from the label list printed in Step 3), then run the same face-detect-or-fallback + CLAHE
  pipeline used in Layer 1.


## Task Summary — L2_T1: Skin Condition Data Gathering

**Goal:** Gather and safely store the datasets for Layer 2 (Skin Condition Analysis: acne,
hyperpigmentation, dryness/oiliness, pore visibility, sensitivity), using the same local-first,
archive-once, verify-from-a-fresh-mount storage pattern the Layer 1 series was rebuilt around.

**Process:**
1. Before building anything, re-verified the Layer 2 dataset research and found it had errors:
   `khanusa/facial-skin-conditions` (named as the primary dataset) does not exist, and a second candidate
   (`killa92` on Kaggle) could not be verified. Both were dropped. ACNE04 was also corrected: it is not on
   Kaggle, and the authors host it on Google Drive / Baidu under an academic-use-only license.
2. Wrote a data provenance notice to Drive recording which licenses are unconfirmed or restrictive, so
   "free to download" isn't mistaken for "cleared for commercial use."
3. Loaded DermNet through its Hugging Face mirror (`rkdeva/Dermnet-Train`), and printed its real schema and
   full label list before exporting anything.
4. Exported all 15,297 images plus a labels CSV to local disk, archived them as one `.tar.gz`, uploaded it to
   Drive, and verified the byte size matched.
5. Forced a `flush_and_unmount` + remount and re-verified from the fresh mount, then opened the archive and
   confirmed its contents before writing the manifest last.

**Accomplished:**
- **DermNet: 15,297 images safely archived on Drive** (1,559.6 MB), verified after a fresh remount. The
  archive contains 15,297 `.jpg` files and 1 labels CSV, with no truncation.
- **Provenance notice and manifest written**, recording the dataset correction, the unverified DermNet
  license, and the not-yet-downloaded status of the manual datasets.
- **A real finding about the labels:** this mirror has **865 distinct labels**, not the ~23 disease
  categories described for the original DermNet. The labels are fine-grained and uneven: only about 100
  labels have 50 or more images, roughly 200 have exactly one, and about 460 have five or fewer. Naming is
  inconsistent (for example `acne closed comedo` vs. `acne Closed Comedo`, and `Acne` vs. `acne`), and one
  label, `IMG` (119 images), carries no diagnostic meaning.
- **Usable signal for Aura's conditions exists, but it is thin.** Acne-related labels span about 15 variants
  and roughly 410 images (`acne cystic` 120, `acne open comedo` 70, `acne pustular` 69). Redness-related
  labels include rosacea (~195 images across 14 labels) and perioral dermatitis (~116). Pigmentation is the
  weakest: `lentigo adults` 73, `sun damaged skin` 99, melasma ~27, freckles 1.

**What was not included, and why:**
- **The four manual datasets (ACNE04, Oily/Dry/Normal skin types, SkinCon, DDI)** — none downloaded yet.
  Each is gated or manual (Google Drive/Baidu, Kaggle download, request-based, registration-based), and the
  notebook records them as not-yet-downloaded without blocking on them.
- **Any DermNet test split** — only the train split was loaded; no test split on the mirror was verified.
- **A DermNet license** — the mirror's dataset card states none, so commercial use is unverified.
- **Label-to-condition mapping** — deliberately deferred to `L2_T2`. Which of the 865 labels correspond to
  Aura's five conditions needs real judgment, and DermNet has no label at all for pore visibility.
- **Pore visibility and per-zone annotation** — nothing gathered here covers either, and both are core to how
  Layer 2 is specified. Closing that gap needs in-house, dermatologist-labeled data, the same blocker as
  Layer 1.